# YOLO Fine-Tuning on Colab (Cursor + Colab Extension)

Course project — Part 4: fine-tune YOLOv8 on **distorted** BDD100K images.

## Before you run
1. In Cursor: **Select Kernel → Colab → New Colab Server → GPU (T4)**
2. **Do NOT upload `data/yolo_distorted/` via "Upload to Colab"** — ~360 MB, often disconnects.

### Recommended: Google Drive + zip (reliable)
On your PC (PowerShell):
```powershell
.\scripts\zip_yolo_for_colab.ps1 noise_snr_10db   # one distortion at a time (~275 MB)
```
Upload the zip from `colab_upload/` to **Google Drive** in your browser (resumable).  
Then run **Cell 2b** below to mount Drive and unzip.

Also upload **only** `src/` via *Upload to Colab* (small, ~few hundred KB).

Expected runtime on T4: **~45–90 min** for 300 images × 30 epochs.

In [ ]:
# Cell 1 — GPU check
!nvidia-smi

In [ ]:
# Cell 2 — Project root (EDIT if using Google Drive)
from pathlib import Path
import os
import sys

# Option A: uploaded via "Upload to Colab" into /content
PROJECT_ROOT = Path("/content/DIP Project")

# Option B: Google Drive — uncomment and fix path:
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/DIP Project")

# Option C: flat upload (src + data at /content root)
if not PROJECT_ROOT.exists() and Path("/content/src").exists():
    PROJECT_ROOT = Path("/content")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project not found at {PROJECT_ROOT}. Upload files or fix PROJECT_ROOT."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", PROJECT_ROOT)
print("Contents:", [p.name for p in PROJECT_ROOT.iterdir()])

In [ ]:
# Cell 2b — RECOMMENDED: load dataset from Google Drive zip
# (Skip this cell if you built the dataset on Colab or uploaded without zip)

from pathlib import Path
import zipfile

from google.colab import drive

drive.mount("/content/drive")

# EDIT: path to your zip on Drive (upload via drive.google.com in browser)
DRIVE_ZIP = Path("/content/drive/MyDrive/DIP Project/colab_upload/noise_snr_10db.zip")

PROJECT_ROOT = Path("/content/DIP Project")
DATA_ROOT = PROJECT_ROOT / "data" / "yolo_distorted"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f"Zip not found: {DRIVE_ZIP}\nUpload it to Drive first.")

print(f"Unzipping {DRIVE_ZIP.name} ...")
with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
    zf.extractall(DATA_ROOT)

# Show what we got
for p in sorted(DATA_ROOT.iterdir()):
  if p.is_dir():
    n = len(list(p.rglob("*.jpg")))
    print(f"  {p.name}: {n} images")

In [ ]:
# Cell 3 — Install dependencies
%pip install -q ultralytics opencv-python-headless PyYAML tqdm matplotlib scikit-image numpy

In [ ]:
# Cell 4 — Configuration (edit as needed)
DISTORTION = "noise"       # noise | low_light | jpeg
LEVEL = 10                   # noise: SNR dB | low_light: gamma | jpeg: quality
NUM_TRAIN = 300
NUM_VAL = 50
SEED = 42
EPOCHS = 30
BATCH = 16                   # reduce to 8 if GPU OOM
REBUILD_DATASET = False      # True to rebuild even if dataset exists
SKIP_DATASET_BUILD = False     # True if you uploaded pre-built data/yolo_distorted/

In [ ]:
# Cell 5 — Build YOLO dataset (distorted images + clean GT boxes)
if not SKIP_DATASET_BUILD:
    rebuild_flag = "--rebuild" if REBUILD_DATASET else ""
    !python -m src.yolo_dataset \
        --distortion {DISTORTION} \
        --level {LEVEL} \
        --num-train {NUM_TRAIN} \
        --num-val {NUM_VAL} \
        --seed {SEED} \
        {rebuild_flag} \
        --preview
else:
    print("Skipping dataset build — using uploaded data/yolo_distorted/")

In [ ]:
# Cell 6 — Fine-tune YOLOv8 (~45–90 min on T4)
!python -m src.run_detection \
    --mode finetune \
    --distortion {DISTORTION} \
    --level {LEVEL} \
    --num-train {NUM_TRAIN} \
    --num-val {NUM_VAL} \
    --epochs {EPOCHS} \
    --batch {BATCH} \
    --seed {SEED}

In [ ]:
# Cell 7 — Evaluate: pretrained vs fine-tuned on distorted val set
!python -m src.run_detection \
    --mode finetune-eval \
    --distortion {DISTORTION} \
    --level {LEVEL} \
    --seed {SEED}

In [ ]:
# Cell 8 — Show results
import json
from IPython.display import Image, display
from src.distortions import level_tag

tag = level_tag(DISTORTION, LEVEL)
metrics_path = PROJECT_ROOT / "outputs" / "metrics" / f"detection_finetune_eval_{DISTORTION}_{tag}.json"
plot_path = PROJECT_ROOT / "outputs" / "figures" / f"detection_finetune_{DISTORTION}_{tag}.png"
weights_path = PROJECT_ROOT / "outputs" / "finetune" / f"yolo_{DISTORTION}_{tag}" / "weights" / "best.pt"

if metrics_path.exists():
    m = json.loads(metrics_path.read_text())
    print(f"Pretrained recall: {m['mean_recall_pretrained']:.3f}")
    print(f"Fine-tuned recall: {m['mean_recall_finetuned']:.3f}")
    print(f"Metrics: {metrics_path}")
else:
    print("Metrics file not found — run finetune-eval first.")

if weights_path.exists():
    print(f"Best weights: {weights_path}")
else:
    print("Weights not found.")

if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print(f"Plot not found: {plot_path}")

## Save results back to your PC

Download these from the Colab server (Explorer → Colab Contents view, or copy to Drive):

- `outputs/finetune/yolo_noise_snr_10db/weights/best.pt`
- `outputs/metrics/detection_finetune_eval_*.json`
- `outputs/figures/detection_finetune_*.png`

Or copy to Drive:
```python
# !cp -r outputs/finetune /content/drive/MyDrive/DIP_results/
```